In [1]:
import pandas as pd
import joblib
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
model = joblib.load("emotion_model.pkl")

tfidf = joblib.load("tfidf.pkl")
destinations = joblib.load("destinations.pkl")


In [3]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [4]:
mood_map = {

    "anger":"Stress Relief",

    "sadness":"Relaxation",

    "fear":"Safe Travel",

    "joy":"Excited",

    "love":"Romantic",

    "surprise":"Adventure"
}

In [5]:
def recommend_trip(user_text):
    cleaned = clean_text(user_text)
    vector = tfidf.transform([cleaned])

    emotion = model.predict(vector)[0]

    probs = model.predict_proba(vector)
    confidence = probs.max()*100

    places = destinations[destinations['Emotion']==emotion]

    return emotion, confidence, places[['Destination','Reason']]

In [11]:
text = input("how was your mood?")
emotion, confidence, emotion_places = recommend_trip(text)

print("\nDetected Emotion:", emotion)
print(f"Confidence Score: {confidence:.2f}%")

print("\nRecommended Destinations:\n")

for _, row in emotion_places.iterrows():
    print(f"📍 {row['Destination']}")
    print(f"   Reason: {row['Reason']}\n")

how was your mood? I want trekking and adventure.



Detected Emotion: drive
Confidence Score: 32.82%

Recommended Destinations:

📍 Leh Ladakh
   Reason: Challenging roads and adventure

📍 Spiti Valley
   Reason: Remote landscapes encouraging exploration

📍 Manali
   Reason: Adventure sports and trekking

📍 Auli
   Reason: Snow sports and outdoor activities

📍 Bir Billing
   Reason: World-famous paragliding destination

📍 Rishikesh
   Reason: Rafting and thrilling experiences

📍 Tawang
   Reason: Offbeat mountain exploration

📍 Kasol
   Reason: Popular backpacking destination

